# Building AI Agent Tools From Scratch

A hands-on notebook that walks through the **full agent loop**:

```
User question
     │
     ▼
┌─────────┐     no tool needed     ┌──────────────┐
│   LLM   │ ─────────────────────► │ Final answer │
└────┬────┘                        └──────────────┘
     │ wants a tool
     ▼
┌─────────┐     result             ┌─────────┐
│  Tool   │ ─────────────────────► │   LLM   │ ──► (repeat until done)
└─────────┘                        └─────────┘
```

**What you'll build:**
1. Plain Python tool functions (`calculate`, `get_weather`)
2. JSON schemas that tell the LLM what tools exist
3. A manual tool-calling flow (step by step)
4. An autonomous agent loop
5. A second agent that analyzes CSV data
6. The same patterns on **Groq** (fast Llama models)
7. Production safety: retries, backoff, and hard stops

> Works in Google Colab or locally. You need an OpenAI API key and (optionally) a Groq API key.


## 1. Setup

Install the OpenAI Python SDK. We use it for **both** OpenAI and Groq — Groq exposes an OpenAI-compatible API, so the same client code works with a different `base_url`.

> **If you see `ModuleNotFoundError: No module named 'openai'`**  
> Run the install cell below **first**, then re-run the import cell.  
> We use `%pip` (not `!pip3`) so the package installs into **the same Python as this notebook kernel**.  
> In Cursor/VS Code: pick a kernel (top-right) — e.g. a project `.venv` — and stick with it.


In [1]:
import json
import os
import time

try:
    from openai import OpenAI
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "openai is not installed in this kernel. "
        "Run the cell above (%pip install), then re-run this cell."
    ) from None

### API keys

Store keys as secrets — never hard-code them in a notebook you might share.

| Provider | Colab secret name | Local alternative |
|---|---|---|
| OpenAI | `openai` | `OPENAI_API_KEY` env var |
| Groq | `groq` | `GROQ_API_KEY` env var |


In [2]:
# trying with completely grok_api

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "❌ No Groq key found. Add GROQ_API_KEY to your .env"
print(f" Groq key loaded. Starts with: {GROQ_API_KEY[:8]}...")

 Groq key loaded. Starts with: gsk_bPiP...


In [4]:
# Two clients, same SDK — different endpoints

client_groq = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

## 2. Your First LLM Call

Before tools, let's confirm the API works. We send a **messages** list — the same format every agent uses.

```
messages = [
  {"role": "system",  "content": "..."},   ← instructions
  {"role": "user",    "content": "..."},   ← the question
]
```

The LLM returns a **completion** — text it generated based on those messages.


In [ ]:
# this is with the openai api key, but we are not using it in this notebook

# SYSTEM_PROMPT = "You are a helpful assistant. Use tools when needed."

# message = [
#     {"role": "system", "content": SYSTEM_PROMPT},
#     {"role": "user", "content": "what is the weather in Hyderabad right now?"},
# ]

# # OpenAI — gpt-4o-mini
# response = client.chat.completions.create(
#     model="gpt-5.5",
#     messages=message, 
# )
# print(response.choices[0].message.content)

I can’t access real-time weather data from here.  

For the current weather in Hyderabad, check a live source like:
- Google: “weather Hyderabad”
- Weather.com
- AccuWeather
- Your phone’s weather app

If you mean **Hyderabad, India**, I can also tell you the typical seasonal weather for this time of year.


In [ ]:
# Peek at the full response object (pretty-printed)
# print(json.dumps(response.choices[0].model_dump(), indent=2))

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "I don\u2019t have access to live weather data right now. For the current weather in Hyderabad, please check a reliable source like Google Weather, Apple Weather, AccuWeather, or the India Meteorological Department.\n\nIf you mean **Hyderabad, Telangana, India**, search: **\u201cHyderabad weather now\u201d**.",
    "refusal": null,
    "role": "assistant",
    "annotations": [],
    "audio": null,
    "function_call": null,
    "tool_calls": null
  }
}


### Same question, Groq (Llama)

Groq hosts open models with very low latency. The API shape is identical — only the model name changes.


In [5]:
SYSTEM_PROMPT = "You are a helpful assistant. Use tools when needed."

message = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "what is the weather in Hyderabad right now?"},
]

In [6]:
response_groq = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=message,
)
print(response_groq.choices[0].message.content)

I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways for you to find out the current weather in Hyderabad:

1. **Check online weather websites**: You can visit websites like AccuWeather, Weather.com, or the Indian Meteorological Department's website to get the current weather conditions in Hyderabad.
2. **Use a weather app**: Download a weather app on your smartphone, such as Dark Sky, Weather Underground, or The Weather Channel, to get real-time weather updates for Hyderabad.
3. **Search on Google**: Simply type "Hyderabad weather" or "current weather in Hyderabad" in Google, and you'll get the latest weather updates.

Please note that the weather can change rapidly, so it's always a good idea to check multiple sources for the most up-to-date information.

If you want to know the general climate and weather patterns in Hyderabad, I can provide you with that information. Hyderabad has a hot and dry climate, with three main

In [7]:
# Peek at the Groq response object (pretty-printed)
print(json.dumps(response_groq.choices[0].model_dump(), indent=2))

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways for you to find out the current weather in Hyderabad:\n\n1. **Check online weather websites**: You can visit websites like AccuWeather, Weather.com, or the Indian Meteorological Department's website to get the current weather conditions in Hyderabad.\n2. **Use a weather app**: Download a weather app on your smartphone, such as Dark Sky, Weather Underground, or The Weather Channel, to get real-time weather updates for Hyderabad.\n3. **Search on Google**: Simply type \"Hyderabad weather\" or \"current weather in Hyderabad\" in Google, and you'll get the latest weather updates.\n\nPlease note that the weather can change rapidly, so it's always a good idea to check multiple sources for the most up-to-date information.\n\nIf you want to know the general climate and weather patterns i

## 3. Write the Tool Functions

Tools are just **regular Python functions**. The LLM never runs them directly — *your code* runs them when the LLM asks.

```
LLM says:  "call get_weather(city='Hyderabad')"
Your code: result = get_weather("Hyderabad")
Your code: send result back to LLM
```

We keep two demo tools:
- `calculate` — math expressions
- `get_weather` — fake weather data (no real API needed)


In [9]:
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "Error: only numbers and + - * / ( ) allowed"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


def get_weather(city: str) -> str:
    """Get weather for a city (fake data for the demo)."""
    fake_weather = {
        "hyderabad": "32°C, Partly Cloudy, Humidity: 65%",
        "bangalore": "26°C, Sunny, Humidity: 55%",
        "mumbai": "30°C, Humid, Humidity: 85%",
        "peddapuram": "38°C, Hot, Humidity: 75%",
        "chennai": "34°C, Cloudy, Humidity: 80%",
        "delhi": "25°C, Clear, Humidity: 45%",
        "karimnagar": "33°C, Sunny, Humidity: 60%",
    }
    city_lower = city.lower().strip()
    if city_lower in fake_weather:
        return fake_weather[city_lower]
    return f"Weather data not available for {city}" 

In [10]:
# Quick sanity check
print("Calculator:", calculate("(25 / 4) + 100 * 3"))
print("Weather:   ", get_weather("Karimnagar"))

Calculator: 306.25
Weather:    33°C, Sunny, Humidity: 60%


## 4. Describe Tools as JSON

The LLM can't see your Python functions. You describe them in a **JSON schema** so the model knows:
- what each tool is called
- what it does
- what arguments it expects

```
┌──────────────────┐         ┌─────────────────────┐
│  Python function │  ──►    │  JSON tool schema   │
│  calculate()     │         │  name, description, │
│  get_weather()   │         │  parameters         │
└──────────────────┘         └─────────────────────┘
                                      │
                                      ▼
                              passed to the LLM API
```


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result. Use this for math computation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g. '(25 * 4) + 100'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city in India.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city, e.g. 'Hyderabad'",
                    }
                },
                "required": ["city"],
            },
        },
    },
]

available_tools = {
    "calculate": calculate,
    "get_weather": get_weather,
}

In [12]:
print("Tools defined:")
for tool in tools:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  🔧 {fn['name']}({', '.join(params)}) — {fn['description']}")

Tools defined:
  🔧 calculate(expression) — Evaluate a mathematical expression and return the result. Use this for math computation.
  🔧 get_weather(city) — Get current weather for a city in India.


## 5. Call the LLM *With* Tools

Pass `tools=tools` to the API. The LLM can now respond in two ways:

| `finish_reason` | Meaning |
|---|---|
| `"stop"` | Final text answer — no tool needed |
| `"tool_calls"` | LLM wants you to run a tool first |

When it picks a tool, the response includes **which function** to call and **what arguments** to pass.


In [ ]:
message = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is 16 + 1 /3 ?"},
]

response = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",  # try with -instant
    messages=message,
    tools=tools,  # You are providing your tools.
)
choice = response.choices[0]
print(choice)

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='j8p1hbr33', function=Function(arguments='{"expression":"16 + 1 / 3"}', name='calculate'), type='function')]))


In [23]:
print(f"Finish reason: {choice.finish_reason}")
print(f"Content: {choice.message.content}")
print(f"Tool calls: {choice.message.tool_calls}")

if choice.message.tool_calls:
    tc = choice.message.tool_calls[0]
    print(f"\n── LLM Decision ──")
    print(f"  Tool:      {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")
    print(f"  ID:        {tc.id}")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='h5nsbj66q', function=Function(arguments='{"expression":"16 + 1 / 3"}', name='calculate'), type='function')]

── LLM Decision ──
  Tool:      calculate
  Arguments: {"expression":"16 + 1 / 3"}
  ID:        h5nsbj66q


### Groq with tools

Same message, same tools — Llama on Groq also supports function calling.


In [ ]:
response_groq = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",   # try with the versatile
    messages=message,
    tools=tools,
)
choice_groq = response_groq.choices[0]
print(f"Finish reason: {choice_groq.finish_reason}")
if choice_groq.message.tool_calls:
    tc = choice_groq.message.tool_calls[0]
    print(f"Tool: {tc.function.name}({tc.function.arguments})")
else:
    print(choice_groq.message.content)

Finish reason: tool_calls
Tool: calculate({"expression":"16 + 1 / 3"})


## 6. Execute the Tool & Send Results Back

This is the **hand-off** step — the part that makes it an *agent*, not just a chatbot.

```
  Step 1: LLM returns tool_calls
              │
              ▼
  Step 2: YOUR code runs the Python function
              │
              ▼
  Step 3: Append tool result to messages
              │
              ▼
  Step 4: Call LLM again → final answer
```

The message history grows like this:

```
[system, user, assistant+tool_calls, tool_result]  →  final answer
```


In [26]:
# Step 2 — parse what the LLM asked for
tool_call = choice.message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print(f"Calling: {function_name}({arguments})")
result = available_tools[function_name](**arguments)
print(f"Result:  {result}")

Calling: calculate({'expression': '16 + 1 / 3'})
Result:  16.333333333333332


In [28]:
# Step 3 — append assistant message + tool result
message.append(choice.message)
message.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result,
})

# Step 4 — ask LLM again with the tool result
final_response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=message,
    tools=tools,
)
print(final_response.choices[0].message.content)

The result of the expression 16 + 1/3 is 16.333333333333332.


In [29]:
# Full conversation so far
for i, msg in enumerate(message):
    role = msg["role"] if isinstance(msg, dict) else msg.role
    print(f"{i}. {role}")

0. system
1. user
2. assistant
3. tool
4. assistant
5. tool


## 7. Manual Flow — All Steps Together

Same flow as above, in one cell. Useful when debugging.


In [30]:
message = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is 1547 * 23 + 89 ?"},
]

# Round 1 — LLM picks a tool
response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile", messages=message, tools=tools,
)
choice = response.choices[0]

tool_call = choice.message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)
result = available_tools[function_name](**arguments)

print(f"🔧 {function_name}({arguments}) → {result}")

message.append(choice.message)
message.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

# Round 2 — LLM writes the final answer
final_response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile", messages=message, tools=tools,
)
print(f"\n✅ Answer: {final_response.choices[0].message.content}")

🔧 calculate({'expression': '1547 * 23 + 89'}) → 35670

✅ Answer: The result of the mathematical expression 1547 * 23 + 89 is 35670.


### Message flow diagram

```
┌────────┐   ┌──────┐   ┌───────────┐   ┌────────┐   ┌───────────┐
│ system │ → │ user │ → │ assistant │ → │  tool  │ → │ assistant │
│ prompt │   │ quest│   │ tool_calls│   │ result │   │ final ans │
└────────┘   └──────┘   └───────────┘   └────────┘   └───────────┘
   msg[0]     msg[1]       msg[2]          msg[3]         (new response)
```


## 8. The Autonomous Agent Loop

Instead of running each step by hand, wrap it in a **loop**:

```
while not done and steps < max_steps:
    response = call_llm(messages)
    if no tool needed:
        return answer          ← done!
    else:
        run tool(s)
        append results
        loop again
```

Three cases the loop handles:
1. **`finish_reason == "stop"`** — LLM has a final answer
2. **`tool_calls` present** — run tools, continue looping
3. **`max_steps` reached** — hard stop, give up


In [ ]:
def run_agent(user_message, max_steps=5, verbose=True):
    """Simple agent loop — LLM + tools until done or max_steps."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the available tools to answer questions accurately. If you can answer without tools, do so directly."},
        {"role": "user", "content": user_message},
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"🧑 User: {user_message}")
        print(f"{'═' * 60}")

    for step in range(max_steps):
        response = client_groq.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
        )
        choice = response.choices[0]

        # Case 1: done
        if choice.finish_reason == "stop":
            if verbose:
                print(f"\n✅ Step {step + 1}: Final answer ready")
            return choice.message.content

        # Case 2: tool call(s)
        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n🔧 Step {step + 1}: {func_name}({args})")

                if func_name in available_tools:
                    result = available_tools[func_name](**args)
                else:
                    result = f"Error: Unknown tool '{func_name}'"

                if verbose:
                    print(f"   📤 Result: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

    # Case 3: hard stop
    return "⚠️ Agent reached maximum steps without finishing."

print("✅ Agent function defined!")

✅ Agent function defined!


### Try it — three scenarios

| Test | Expected behavior |
|---|---|
| Math question | Calls `calculate`, returns number |
| Weather question | Calls `get_weather`, returns forecast |
| "What is Python?" | No tool — LLM answers directly |


In [35]:
answer = run_agent("What is (45 * 67) + (23 * 89)?")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What is (45 * 67) + (23 * 89)?
════════════════════════════════════════════════════════════

🔧 Step 1: calculate({'expression': '(45 * 67) + (23 * 89)'})
   📤 Result: 5062

✅ Step 2: Final answer ready

🤖 Answer: The result of the expression \((45 * 67) + (23 * 89)\) is 5062.


In [36]:
answer = run_agent("What's the weather like in Hyderabad?")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the weather like in Hyderabad?
════════════════════════════════════════════════════════════

🔧 Step 1: get_weather({'city': 'Hyderabad'})
   📤 Result: 32°C, Partly Cloudy, Humidity: 65%

✅ Step 2: Final answer ready

🤖 Answer: The weather in Hyderabad is currently 32°C, partly cloudy, with a humidity level of 65%.


In [37]:
answer = run_agent("What is Python?")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What is Python?
════════════════════════════════════════════════════════════

✅ Step 1: Final answer ready

🤖 Answer: Python is a high-level, interpreted programming language known for its readability and simplicity. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Key features of Python include:

1. **Readability**: Python's syntax is clear and straightforward, making it easy for developers to read and understand code.
2. **Dynamic Typing**: Variables in Python do not require an explicit declaration to reserve memory space. The declaration happens automatically when a value is assigned to a variable.
3. **Comprehensive Standard Library**: Python comes with a large standard library that provides modules and functions for various tasks, such as file I/O, system calls, and even Internet prot

> **Without tools**, the LLM guesses from training data. With tools, it *acts* — that's the difference between a chatbot and an agent.


In [ ]:
# Without tools — LLM can only guess
message = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather like in Hyderabad?"},
]
response = client_groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=message)
print(response.choices[0].message.content)

I currently don't have the ability to access real-time weather information. However, you can easily find the current weather in Hyderabad by checking a weather website or using a weather app on your smartphone. Popular options include the Weather Channel, AccuWeather, or even a quick search on Google for "Hyderabad weather." If you're looking for historical data or general weather patterns, feel free to ask!


## 9. Build Another Agent — CSV Data Analyst

Same pattern, different tools. The agent gets a small employee dataset and three analysis tools.

```
Question: "Average salary?"
     │
     ▼
 list_columns() ──► get_stats("salary") ──► final answer
```

The loop code is identical — only the **tools** and **system prompt** change.


In [39]:
import csv
from io import StringIO

CSV_DATA = """name,department,salary,experience_years,city
Rahul,Engineering,75000,5,Hyderabad
Priya,Marketing,55000,3,Bangalore
Arjun,Engineering,82000,7,Hyderabad
Sneha,HR,48000,2,Mumbai
Vikram,Engineering,90000,9,Delhi
Anita,Marketing,60000,4,Bangalore
Karthik,HR,52000,3,Chennai
Deepa,Engineering,70000,4,Hyderabad
Suresh,Marketing,65000,6,Mumbai
Kavya,HR,50000,2,Bangalore
Ravi,Engineering,95000,10,Delhi
Meera,Marketing,58000,3,Chennai
Arun,Engineering,78000,6,Hyderabad
Lavanya,HR,55000,4,Bangalore
Sanjay,Marketing,62000,5,Mumbai
"""

reader = csv.DictReader(StringIO(CSV_DATA.strip()))
data = list(reader)

print(f"Dataset: {len(data)} employees")
print(f"Columns: {list(data[0].keys())}")
print("\nFirst 3 rows:")
for row in data[:3]:
    print(f"  {row}")

Dataset: 15 employees
Columns: ['name', 'department', 'salary', 'experience_years', 'city']

First 3 rows:
  {'name': 'Rahul', 'department': 'Engineering', 'salary': '75000', 'experience_years': '5', 'city': 'Hyderabad'}
  {'name': 'Priya', 'department': 'Marketing', 'salary': '55000', 'experience_years': '3', 'city': 'Bangalore'}
  {'name': 'Arjun', 'department': 'Engineering', 'salary': '82000', 'experience_years': '7', 'city': 'Hyderabad'}


In [40]:
def list_columns() -> str:
    """List all columns with sample values."""
    lines = []
    for col in data[0].keys():
        samples = [row[col] for row in data[:3]]
        lines.append(f"{col}: {samples}")
    return "\n".join(lines)


def get_stats(column: str) -> str:
    """Get mean, min, max, count for a numeric column."""
    try:
        values = [float(row[column]) for row in data]
        return json.dumps({
            "column": column,
            "count": len(values),
            "mean": round(sum(values) / len(values), 2),
            "min": min(values),
            "max": max(values),
            "total": sum(values),
        })
    except (ValueError, KeyError) as e:
        return f"Error: {e}. Try: salary, experience_years"


def filter_count(column: str, value: str) -> str:
    """Count rows where column equals value."""
    try:
        matches = [row for row in data if row[column].lower() == value.lower()]
        if matches:
            return json.dumps({
                "filter": f"{column} = {value}",
                "count": len(matches),
                "matching_names": [m["name"] for m in matches],
            })
        return f"No rows found where {column} = {value}"
    except KeyError:
        return f"Error: Column '{column}' not found."

# Quick test
print(list_columns())
print()
print(get_stats("salary"))
print()
print(filter_count("department", "Engineering"))

name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']

{"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}

{"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}


In [41]:
csv_tools = [
    {
        "type": "function",
        "function": {
            "name": "list_columns",
            "description": "List all columns in the employee dataset with sample values. Use this first to understand what data is available.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stats",
            "description": "Get statistics (mean, min, max, count, total) for a numeric column like 'salary' or 'experience_years'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Numeric column name"},
                },
                "required": ["column"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "filter_count",
            "description": "Count rows where column equals value. Returns count and matching employee names.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Column to filter on"},
                    "value": {"type": "string", "description": "Value to match"},
                },
                "required": ["column", "value"],
            },
        },
    },
]

csv_available_tools = {
    "list_columns": list_columns,
    "get_stats": get_stats,
    "filter_count": filter_count,
}

print("CSV Analyst tools ready:")
for t in csv_tools:
    print(f"  🔧 {t['function']['name']}")

CSV Analyst tools ready:
  🔧 list_columns
  🔧 get_stats
  🔧 filter_count


In [42]:
def csv_analyst(question, verbose=True):
    """Agent that answers questions about the employee dataset."""
    messages = [
        {"role": "system", "content": (
            "You are a data analyst assistant. You have access to an employee dataset. "
            "Use the available tools to answer questions. "
            "Give clear, concise answers with the numbers."
        )},
        {"role": "user", "content": question},
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"📊 Question: {question}")
        print(f"{'─' * 60}")

    max_steps = 5
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=csv_tools,
        )
        choice = response.choices[0]

        if choice.finish_reason == "stop":
            if verbose:
                print(f"{'─' * 60}")
            return choice.message.content

        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    args_str = ", ".join(f"{k}={v}" for k, v in args.items())
                    print(f"  🔧 {func_name}({args_str})")

                result = csv_available_tools.get(func_name, lambda **_: f"Unknown tool: {func_name}")(**args)

                if verbose:
                    print(f"     → {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

    return "Could not answer within step limit."

print("✅ CSV Analyst Agent ready!")

✅ CSV Analyst Agent ready!


In [43]:
answer = csv_analyst("What is the average salary in the company?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: What is the average salary in the company?
────────────────────────────────────────────────────────────
  🔧 get_stats(column=salary)
     → {"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}
────────────────────────────────────────────────────────────

🤖 The average salary in the company is $66,333.33.


In [44]:
answer = csv_analyst("How many people work in Engineering? Who are they?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: How many people work in Engineering? Who are they?
────────────────────────────────────────────────────────────
  🔧 filter_count(column=department, value=Engineering)
     → {"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}
────────────────────────────────────────────────────────────

🤖 There are 6 people working in Engineering. Their names are:
- Rahul
- Arjun
- Vikram
- Deepa
- Ravi
- Arun


In [47]:
answer = csv_analyst("Which department has higher salaries — Engineering or Marketing? By how much?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: Which department has higher salaries — Engineering or Marketing? By how much?
────────────────────────────────────────────────────────────
  🔧 filter_count(column=department, value=Engineering)
     → {"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}
  🔧 filter_count(column=department, value=Marketing)
     → {"filter": "department = Marketing", "count": 5, "matching_names": ["Priya", "Anita", "Suresh", "Meera", "Sanjay"]}
  🔧 get_stats(column=salary)
     → {"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}
  🔧 filter_count(column=department, value=Engineering)
     → {"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}
  🔧 filter_count(column=department, value=Marketing)
     → {"filter": "department = Marketing", "co

In [46]:
answer = csv_analyst("How many employees are based in Hyderabad?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: How many employees are based in Hyderabad?
────────────────────────────────────────────────────────────
  🔧 filter_count(column=location, value=Hyderabad)
     → Error: Column 'location' not found.
  🔧 list_columns()
     → name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']
  🔧 filter_count(column=city, value=Hyderabad)
     → {"filter": "city = Hyderabad", "count": 4, "matching_names": ["Rahul", "Arjun", "Deepa", "Arun"]}
────────────────────────────────────────────────────────────

🤖 There are 4 employees based in Hyderabad: Rahul, Arjun, Deepa, and Arun.


## 10. Safety — Retries, Backoff & Hard Stops

In production, things go wrong:
- API rate limits (too many requests)
- Network timeouts
- LLM getting stuck in loops

We need **three safety mechanisms**:

| Safety Net | What It Does | Analogy |
|---|---|---|
| **Retry** | Try again if it fails | Redialing a busy phone |
| **Backoff** | Wait longer between retries | Waiting 1s, then 2s, then 4s... |
| **Hard Stop** | Quit after N attempts | "If no answer in 5 tries, give up" |

```
API call fails
     │
     ▼
  Retry #1 ──wait 1s──► Retry #2 ──wait 2s──► Retry #3 ──► give up
                                                          (hard stop)
```

We already use a **hard stop** via `max_steps` in the agent loop. Now let's add **retry + backoff** around the API call itself.


In [48]:
def call_llm_with_retry(client, model, messages, tools=None, max_retries=3):
    """
    Call the LLM API with retry + exponential backoff.

    Waits 1s, then 2s, then 4s between attempts.
    Gives up after max_retries (hard stop).
    """
    for attempt in range(max_retries):
        try:
            kwargs = {"model": model, "messages": messages}
            if tools:
                kwargs["tools"] = tools
            return client.chat.completions.create(**kwargs)
        except Exception as e:
            if attempt == max_retries - 1:
                raise  # hard stop — no more retries
            wait = 2 ** attempt  # 1s, 2s, 4s ...
            print(f"⚠️  Attempt {attempt + 1} failed: {e}")
            print(f"   Retrying in {wait}s...")
            time.sleep(wait)

In [49]:
def run_agent_safe(user_message, max_steps=5, max_retries=3, verbose=True):
    """Production-ready agent: retry/backoff on API errors + hard stop on loop count."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user", "content": user_message},
    ]

    if verbose:
        print(f"\n🛡️ Safe agent: {user_message}")

    for step in range(max_steps):  # hard stop on loop
        response = call_llm_with_retry(
            client, "gpt-4o-mini", messages, tools=tools, max_retries=max_retries,
        )
        choice = response.choices[0]

        if choice.finish_reason == "stop":
            return choice.message.content

        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                if verbose:
                    print(f"  🔧 {func_name}({args})")
                result = available_tools.get(func_name, lambda **_: "Unknown tool")(**args)
                messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "⚠️ Agent reached maximum steps without finishing."

# Works the same as run_agent — but survives transient API errors
answer = run_agent_safe("What is 1547 * 23 + 89?")
print(f"\n🤖 {answer}")


🛡️ Safe agent: What is 1547 * 23 + 89?
  🔧 calculate({'expression': '1547 * 23 + 89'})

🤖 The result of \( 1547 \times 23 + 89 \) is 35,670.


### Three layers of safety (stacked)

```
┌─────────────────────────────────────────────────────────┐
│  Layer 3: Hard Stop (max_steps)                         │
│  "Don't loop forever if the LLM keeps calling tools"    │
├─────────────────────────────────────────────────────────┤
│  Layer 2: Backoff (1s → 2s → 4s)                        │
│  "Don't hammer the API when it's overloaded"            │
├─────────────────────────────────────────────────────────┤
│  Layer 1: Retry (up to N attempts)                      │
│  "Transient errors often succeed on the next try"       │
└─────────────────────────────────────────────────────────┘
```


## 12. Summary — What You Built

You went from a plain LLM call to a **production-shaped agent**:

```
┌──────────────────────────────────────────────────────────────┐
│                     THE AGENT STACK                          │
├──────────────────────────────────────────────────────────────┤
│  1. Tool functions     calculate(), get_weather(), ...       │
│  2. JSON schemas       tell the LLM what tools exist         │
│  3. Tool execution     YOUR code runs what the LLM requests  │
│  4. Message history    system → user → assistant → tool → …  │
│  5. Agent loop         repeat until done or max_steps          │
│  6. Provider swap      OpenAI ↔ Groq (same SDK, diff model)  │
│  7. Safety nets        retry + backoff + hard stop           │
└──────────────────────────────────────────────────────────────┘
```

### Key ideas to remember

| Concept | One-liner |
|---|---|
| **Tool** | A Python function the LLM can *request* but not *run* |
| **Tool schema** | JSON description so the LLM knows name + arguments |
| **Agent loop** | Call LLM → run tools → call LLM again → repeat |
| **finish_reason** | `"stop"` = done, `"tool_calls"` = needs your code |
| **Hard stop** | `max_steps` prevents infinite loops |
| **Retry + backoff** | Survive rate limits and network blips |

### The whole loop in one picture

```
     ┌──────────────────────────────────────────┐
     │              AGENT LOOP                   │
     │                                          │
     │   User ──► LLM ──► tool? ──yes──► Run fn  │
     │              ▲                    │      │
     │              └──── result ────────┘      │
     │              │                           │
     │              └── no tool ──► Answer      │
     │                                          │
     │   Safety: retry/backoff on API errors    │
     │           max_steps hard stop on loops   │
     └──────────────────────────────────────────┘
```

That's it — every AI agent framework (LangChain, CrewAI, OpenAI Assistants, etc.) is built on this same loop. You now understand what's happening under the hood.
